# nerfstudio → 3D ULPIN reconstruction (one-time GPU run)

Run **once on a CUDA Colab VM / GPU box** to rebuild the synthetic locality with
real nerfstudio `splatfacto`, then download the artifacts so the demo runs
fully offline (the FastAPI backend never trains).

Steps:
1. Check GPU
2. Install nerfstudio + tiny-cuda-nn
3. Generate synthetic drone imagery (or upload real drone video/images)
4. `ns-process-data` → `ns-train splatfacto` → `ns-export pointcloud`
5. Georeference (GCP or pose-seed) and write the manifest
6. Download `data/reconstruction/` back into the repo

In [ ]:
!nvidia-smi

In [ ]:
import sys, os
print("python", sys.version.split()[0])
# mount the repo so the app scripts + synthetic data are reachable
sys.path.insert(0, "/content")

## 1. Install nerfstudio (≈5-10 min)

Works on CUDA 11.8 / 12.x. On Colab we build tiny-cuda-nn from source.

In [ ]:
%%capture
!pip install --upgrade pip setuptools
!pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install ninja
!git clone --depth 1 https://github.com/NVlabs/tiny-cuda-nn /content/tcnn || true
!pip install git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch
!pip install nerfstudio
import nerfstudio; print("nerfstudio", nerfstudio.__version__)

## 2. Get the project

Either `git clone` the repo or upload `backend/` + `data/` into `/content/3dulpin`. If
you only have real drone images, put them in `data/raw/drone_views/` (any video
works too) and skip the synthetic generator.

In [ ]:
%%capture
%cd /content
!git clone https://github.com/<your-org>/3dulpin /content/3dulpin 2>/dev/null || true
%cd /content/3dulpin || exit 1

## 3. Generate the synthetic drone imagery (drone-views)

Renders 24 perspective aerial views + UTM camera poses (`transforms.json`) —
this is the *drone imagery* input for reconstruction. Safe CoLab scene (~200 m),
a few hundred ms per frame.

In [ ]:
!python -m venv /tmp/py && /tmp/py/bin/pip install -q rasterio pillow numpy 2>/dev/null || true
PYTHONPATH=backend python backend/app/scripts/generate_drone_views.py
!ls data/raw/drone_views | head

## 4. Reconstruct with splatfacto (this is the GPU-bound step)

`--skip-colmap` because `transforms.json` already carries UTM-seeded poses.
`--pose-seed` records the (near-identity) georeference. For real captures pass
`--gcp-json` with survey/GCP control pairs instead, and drop `--skip-colmap`.

In [ ]:
PYTHONPATH=backend python backend/app/scripts/reconstruct_gpu.py \
  --data data/raw/drone_views \
  --method splatfacto \
  --max-iter 20000 \
  --skip-colmap \
  --pose-seed \
  --data-dir data

## 5. Verify + download artifacts

Check the manifest and download `data/reconstruction/` into your local repo.

In [ ]:
import json
m = json.load(open("data/reconstruction/manifest.json"))
print(json.dumps(m, indent=2))
from google.colab import files
!zip -qr /content/reconstruction.zip data/reconstruction
files.download("/content/reconstruction.zip")

## 6. Back on your laptop

Unzip into the repo root — `data/reconstruction/{manifest.json, pointcloud.ply, dsm.tif,…}`.
Then the app will automatically prefer the nerfstudio artifact:

```bash
unzip /path/to/reconstruction.zip -d .
curl -X POST "http://localhost:8000/api/pipeline/run?stage=reconstruct"
```

The reconstruction stage now reads `pointcloud.ply` instead of `pointcloud.las`.

## 7. (Optional) Photoreal building from real photos → 3D viewer

For a **real building captured with a phone/drone**, this runbook converts the trained
splatfacto model into the browser `.splat` that the 3D scene renders on the cadastral
footprint:

```bash
# 1. Train on your own photos (COLMAP poses, GCP-constrained):
#    run reconstruct_gpu.py with real frames + --gcp-json, WITHOUT --pose-seed

# 2. Export the gaussian splat model as an INRIA-style PLY:
ns-export gaussian-splat --load-config <config.yml> --output-dir exports/splat

# 3. Convert PLY → .splat, georeference to the footprint, register:
PYTHONPATH=backend python backend/app/scripts/register_real_building.py \
  --building-id 253 \
  --input exports/splat/point_cloud/iteration_<N>/point_cloud.ply \
  --gcps data/raw/site/gcps.json   # [{local:[x,y,z], target:[X,Y,Z]}] 3+ pairs
# writes data/reconstruction/real/253/{scene.splat, pose.json}

# 4. Reload the scene — selecting the building shows it photoreal in-view
#    (GET /api/focus/building/253 returns .splat url + pose).
```

Upstream projects that make this possible: **nerfstudio-project/nerfstudio** (training),
**graphdeco-inria/gaussian-splatting** (reference 3DGS), **antimatter15/splat**
(.ply↔.splat converter reference) and **pmndrs/drei Splat** (browser renderer).